[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Exploring an API


## What you will be able to do

Try an API by hand before writing code against it: send requests with curl from a notebook cell,
read an API's documentation and its OpenAPI document, and know what Postman, Bruno and interactive
documentation pages do for you.


## The idea

### The problem

You have an API's address and a job to do: fetch Tromso's daily temperatures in Fahrenheit, say.
The quick-looking way is to start writing Python and adjust it until something works. Every guess
costs a run, and the guesses that fail loudly are the lucky ones.

The unlucky guess succeeds. Ask Open-Meteo for Fahrenheit with `unit=fahrenheit` and the response
comes back `200 OK`, with the temperatures in Celsius. Open-Meteo has no parameter called `unit`,
so it ignored it, as most APIs do with parameters they do not recognize, and nothing in the
response says so. The parameter is `temperature_unit`, and the documentation lists it.

So before writing code against an API, people explore it. They read what it accepts and what it
returns, and they send a few requests by hand to see real responses. That takes minutes, and it is
where the facts the code depends on come from.

### What documentation is

> An API's **documentation** describes what it accepts and what it returns: the base URL, the
> endpoints and their methods, the parameters each endpoint takes, the responses it sends, and the
> rules around them, such as authentication and rate limits. An **OpenAPI document** is the same
> description in a standard, machine-readable form: a JSON or YAML file that tools read to build
> requests, generate code, and draw interactive documentation.

### The tools

Four kinds of tool do the exploring, from the simplest:

| Tool | What it does | Where it runs |
|---|---|---|
| A browser | Opening a URL sends a `GET` request and shows the response body. The Network tab of its developer tools lists every request a page makes, with status codes and headers | on your computer |
| curl | A command-line client: every part of a request is a flag, and it prints the response | macOS, Linux and Windows, and in a notebook cell after `!` |
| Postman, Bruno, Insomnia | Graphical clients: build a request in a form, save requests in collections, import an OpenAPI document, and turn any request into code | on your computer; Postman also in a browser |
| Interactive documentation | A web page drawn from an OpenAPI document, such as Swagger UI, where a button sends a request from the page | in a browser, wherever the API publishes it |

### Why it works that way

Every one of these tools sends the same HTTP requests Python will. A request that works in curl or
Postman works from Python, and a request that fails there fails in Python too, with less code in
the way of seeing why. That makes a request sent by hand the fastest test of an assumption about an
API.

curl has a second role: it is how documentation writes requests down. A curl command is a complete
request on a single line, with nothing implied, so API documentation uses it for examples, and
Postman and interactive documentation pages will hand you any request as a curl command. You will
read curl far more often than you write it.

### Where this shows up

GitHub's REST API documentation shows a curl command for every endpoint. Open-Meteo's documentation
builds the request URL for you as you tick options. Swagger's Petstore is interactive documentation
you can try in a browser. Postman is common in teams that build APIs. In this guide, the **Your
First API Server** notebook generates an OpenAPI document and interactive documentation for an API
of your own.

### The practice API's documentation

The practice API now publishes its own OpenAPI document, at `/openapi.json`. Postman cannot reach
the practice API while it runs inside Colab, because it listens on Colab's computer, not yours. To
explore it with Postman, download
[`practice_api.py`](https://github.com/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/practice_api.py)
and run `python practice_api.py` on your own computer: it then answers at the same address there.

### What this notebook covers

- curl from a notebook cell: a response with its headers, another method, a query, a saved file
- The practice API's OpenAPI document read with Python, and a `$ref` followed to its schema
- Open-Meteo's documentation, read for what code needs to know
- Postman, Bruno, interactive documentation and a browser's developer tools, as steps to follow
- An OpenAPI document turned into requests, the way Postman's import does it
- Six errors, including a request that failed and printed no error at all, and a parameter
  Open-Meteo ignored

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
!curl -i http://127.0.0.1:8765/stations/tromso
```

```
HTTP/1.1 200 OK
Server: PracticeAPI/1.0
Date: Sun, 01 Mar 2026 09:00:00 GMT
Content-Type: application/json
Content-Length: 73

{"id": "tromso", "name": "Tromso", "latitude": 69.65, "longitude": 18.96}
```

That is the `GET` request that took a socket and a helper function in the **What an API Is**
notebook, as a single command. The `!` hands the line to the system's shell instead of to Python,
and `-i` asks curl to include the response's status line and headers; without it, curl prints only
the body.


## Setup

Seven imports, the last of them the practice API.

- `json` turns response bodies, and the OpenAPI document, into Python values
- `urllib.request` fetches the practice API's code in Colab, and sends requests from Python
- `Path` checks whether the practice API's code is already here, and reads a file curl saves
- `HTTPError` catches an error status, which `urlopen` raises as an exception
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`.
  `open_meteo()` returns Open-Meteo's address, or the address of the practice API's recording of
  it when Open-Meteo is not answering

If Open-Meteo stops answering while you work through the notebook, run this cell again: it checks
again, and the Open-Meteo cells switch to the recording.


In [1]:
import importlib
import json
import sys
import urllib.request
from pathlib import Path
from urllib.error import HTTPError

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()
print("The practice API is running at", BASE)
print("Open-Meteo's archive is at", OPEN_METEO)


The practice API is running at http://127.0.0.1:8765
Open-Meteo's archive is at https://archive-api.open-meteo.com/v1/archive


## Worked examples

### curl from a notebook cell

A line that starts with `!` runs as a shell command, and curl is installed in Colab, on macOS and
most Linux systems, and on Windows 10 and later. Inside the line, `{BASE}` is replaced with the
value of the Python variable `BASE` before the command runs. `-i` includes the response's status
line and headers:


In [2]:
!curl -i {BASE}/stations/tromso


HTTP/1.1 200 OK
Server: PracticeAPI/1.0
Date: Sun, 01 Mar 2026 09:00:00 GMT
Content-Type: application/json
Content-Length: 73

{"id": "tromso", "name": "Tromso", "latitude": 69.65, "longitude": 18.96}

That is the response the **What an API Is** notebook read over a raw connection: the status line,
the headers, a blank line, and the body. Without `-i`, curl prints only the body. `-s`, for silent,
stops curl printing a progress meter when its output goes somewhere other than the screen, such as
into another program. Here it goes into `python3 -m json.tool`, which lays JSON out with
indentation:


In [3]:
!curl -s {BASE}/stations | python3 -m json.tool


[
    {
        "id": "bergen",
        "name": "Bergen"
    },
    {
        "id": "oslo",
        "name": "Oslo"
    },
    {
        "id": "svalbard",
        "name": "Svalbard"
    },
    {
        "id": "tromso",
        "name": "Tromso"
    }
]


`-X` sets the method. Here is the `DELETE` request from the **What an API Is** notebook, as a
single command:


In [4]:
!curl -i -X DELETE {BASE}/stations/tromso


HTTP/1.1 405 Method Not Allowed
Server: PracticeAPI/1.0
Date: Sun, 01 Mar 2026 09:00:00 GMT
Content-Type: application/json
Content-Length: 59
Allow: GET

{"error": "DELETE not allowed: the stations are read-only"}

Two more flags appear constantly in documentation: `-H` adds a request header, as in
`-H "Accept: application/json"`, and `-d` sends a request body, which makes curl use `POST` unless
`-X` says otherwise. The **Sending Data** notebook uses both. In Windows PowerShell, type
`curl.exe`, because `curl` alone runs PowerShell's own `Invoke-WebRequest`, which takes different
options.

### A query, sent with curl and saved to a file

A real query reads best with a parameter per line. `-G` tells curl to put every `--data-urlencode`
value into the query of a `GET` request, encoded as a URL requires. A backslash at the end of a
line continues the command, `--max-time 30` gives up after 30 seconds, `-o` saves the body to a
file, and `-w` prints the status code once the response has arrived.

`{OPEN_METEO}` is the address Setup chose. Because the line uses a Python variable, curl's own
braces are doubled, `%{{http_code}}`, and IPython hands them on to curl single. The Common errors
section shows what happens when they are not doubled:


In [5]:
!curl -s -G {OPEN_METEO} --max-time 30 \
    --data-urlencode latitude=69.65 --data-urlencode longitude=18.96 \
    --data-urlencode start_date=2025-01-15 --data-urlencode end_date=2025-01-17 \
    --data-urlencode daily=temperature_2m_mean --data-urlencode models=era5 \
    -o tromso.json -w "status %{{http_code}}\n"


status 200


Then Python reads the saved file, which is the step from exploring an API to writing code for it.
The file has done its job once it is read, so the cell removes it.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.


In [6]:
saved = Path("tromso.json")
weather = json.loads(saved.read_text(encoding="utf-8"))
saved.unlink()

print(weather["daily_units"]["temperature_2m_mean"], weather["daily"]["temperature_2m_mean"])


°C [4.6, 6.4, 7.1]


### Reading an OpenAPI document

The practice API publishes its OpenAPI document at `/openapi.json`. It is JSON, so everything from
the **What an API Is** notebook reads it. The top level says what the document describes:


In [7]:
with urllib.request.urlopen(f"{BASE}/openapi.json") as response:
    doc = json.loads(response.read())

print("keys:   ", list(doc))
print("openapi:", doc["openapi"])
print("info:   ", doc["info"])
print("servers:", doc["servers"])


keys:    ['openapi', 'info', 'servers', 'paths', 'components']
openapi: 3.1.0
info:    {'title': 'Practice API', 'version': '1.0', 'description': 'Weather stations, for the APIs and JSON guide.'}
servers: [{'url': 'http://127.0.0.1:8765'}]


`openapi` is the version of the OpenAPI standard the document follows, `info` describes the API,
and `servers` gives the base URL every path is added to. `paths` is the part you will read most:
an endpoint per key, then a method, then what that operation takes and returns.


In [8]:
for path, operations in doc["paths"].items():
    for method, operation in operations.items():
        print(f"{method.upper():<6} {path:<16} {operation['summary']}")
        for p in operation.get("parameters", []):
            print(f"{'':<23} parameter {p['name']!r} in the {p['in']}, {p['schema']['type']}, "
                  f"required: {p['required']}, for example {p['example']!r}")
        for status, response in operation["responses"].items():
            print(f"{'':<23} {status}: {response['description']}")


GET    /stations        List every station
                        200: A summary of every station
GET    /stations/{id}   Get one station
                        parameter 'id' in the path, string, required: True, for example 'tromso'
                        200: The station
                        404: No station has that id


Everything code needs to call the API is here: the endpoints, the parameter a station's id travels
in, and the status codes to expect, `404` included. This is what Postman reads when it imports a
document, and what an interactive documentation page draws.

### Following a $ref to a schema

What does a station look like? The `200` response of `GET /stations/{id}` says what its body
contains, though not directly:


In [9]:
content = doc["paths"]["/stations/{id}"]["get"]["responses"]["200"]["content"]
print(content)


{'application/json': {'schema': {'$ref': '#/components/schemas/Station'}}}


The schema is a **reference**, `$ref`: a pointer to a schema written once, under `components`, and
reused wherever it is needed. The part after `#` is a path through the document, a key at a time.
`resolve` follows it:


In [10]:
def resolve(doc, schema):
    """Return the schema a $ref points to, or the schema itself when there is no $ref."""
    if "$ref" not in schema:
        return schema
    node = doc
    for key in schema["$ref"].removeprefix("#/").split("/"):
        node = node[key]
    return node


station_schema = resolve(doc, content["application/json"]["schema"])

print("type:    ", station_schema["type"])
print("required:", station_schema["required"])
for name, field in station_schema["properties"].items():
    print(f"  {name:<10} {field['type']}")


type:     object
required: ['id', 'name', 'latitude', 'longitude']
  id         string
  name       string
  latitude   number
  longitude  number


That is a **schema**: the shape a station must have, with the type of every field and a list of the
fields that must be present. It matches what the practice API sent back for Tromso. The **Schemas
and Validation** notebook uses documents like this to check that a response has the shape it
promises.

### Reading real documentation: Open-Meteo

Real documentation is written for people, and most of it is a web page. Open-Meteo's archive is
documented at https://open-meteo.com/en/docs/historical-weather-api. Reading it with these
questions in mind answers most of what code needs:

| Question | What Open-Meteo's documentation says |
|---|---|
| What is the base URL? | `https://archive-api.open-meteo.com/v1/archive` |
| Is a key needed? | No, for non-commercial use under 10,000 requests a day |
| Which parameters are required? | `latitude`, `longitude`, `start_date` and `end_date` |
| How are dates written? | ISO 8601, as in `2025-01-15` |
| How is the unit chosen? | `temperature_unit`, which is `celsius` unless set to `fahrenheit` |
| What comes back? | JSON, with units under `daily_units` and values under `daily` |
| What does an error look like? | A `400` with a JSON body holding `error` and `reason` |

Documentation has quirks too. Open-Meteo's says `timezone` is required whenever daily values are
requested. Every request in this guide leaves it out, and the archive answers in GMT, as the
`timezone` field of its response reports. Documentation can be stricter than the server, or looser,
and a request sent by hand shows which.

Here is the request the table describes, for Tromso's temperatures in Fahrenheit:


In [11]:
def daily_means(**extra):
    """Tromso's three daily mean temperatures from Open-Meteo, with extra query parameters."""
    query = {"latitude": 69.65, "longitude": 18.96, "start_date": "2025-01-15",
             "end_date": "2025-01-17", "daily": "temperature_2m_mean", "models": "era5", **extra}
    url = f"{OPEN_METEO}?" + "&".join(f"{name}={value}" for name, value in query.items())
    with urllib.request.urlopen(url, timeout=30) as response:
        body = json.loads(response.read())
    return response.status, body["daily_units"]["temperature_2m_mean"], body["daily"]["temperature_2m_mean"]


print(*daily_means(temperature_unit="fahrenheit"))


200 °F [40.3, 43.5, 44.8]


The status is `200`, and the response reports its unit as `°F`, so Open-Meteo used the parameter.
Reading the unit a response reports, rather than assuming it, is a check code can make for itself,
and the Common errors section shows a request it catches.

A misspelled value is refused, with a `400`:


In [12]:
bad = (f"{OPEN_METEO}?latitude=69.65&longitude=18.96"
       "&start_date=2025-01-15&end_date=2025-01-17&daily=temperature_2m_means")

try:
    urllib.request.urlopen(bad, timeout=30)
except HTTPError as error:
    body = json.loads(error.read())
    print(error.code, error.reason)
    print("error: ", body["error"])
    print("reason:", body["reason"])


400 Bad Request
error:  True
reason: Invalid value: Cannot initialize ForecastVariableDaily from invalid String value temperature_2m_means


The reason names the value it could not use, `temperature_2m_means`, in the server's own terms.
Open-Meteo sends the keys of this body in a different order from one request to the next, which
JSON allows and a dictionary does not mind, and which is why this cell prints fields by name
rather than the raw body.

### Postman, Bruno and interactive documentation

These run outside the notebook, so they are steps to follow rather than cells to run.

**Postman**, which is free for individual use and asks you to sign in for most features, or
**Bruno**, which is open source, needs no account, and saves requests as plain files:

1. Create a request, choose `GET`, and paste `https://archive-api.open-meteo.com/v1/archive` into
   the address box.
2. Open the Params tab and add the six parameters from the curl command above as rows, starting
   with `latitude` and `69.65`. The address box builds the query as you type.
3. Send it. The response panel shows the status code, the time taken, the headers, and the body
   laid out as JSON, with `4.6`, `6.4` and `7.1` in it.
4. Add a row for `temperature_unit` with the value `fahrenheit`, and send again. Then rename the row
   to `unit`, send, and watch the unit return to `°C` with no error.
5. Find the code option, a `</>` icon in Postman: it shows the same request as a curl command or as
   Python, which is how a request explored by hand becomes code.
6. Import an OpenAPI document to get a request for every endpoint at once. Swagger's Petstore
   publishes one at `https://petstore3.swagger.io/api/v3/openapi.json`. That document gives its
   server as the relative address `/api/v3`, so if the imported base URL shows only that, set it to
   `https://petstore3.swagger.io/api/v3`.

**Interactive documentation.** Open https://petstore3.swagger.io in a browser. Every endpoint in
its OpenAPI document is listed. Open an endpoint, choose Try it out, fill in the parameters, and
choose Execute: the page shows the curl command it sent, the response, and the schema of the body.

**A browser's developer tools.** On any website, open the developer tools, with F12, or
Cmd+Option+I on a Mac, and choose the Network tab. Reloading the page lists every request it makes,
many of them to APIs, with their methods, status codes, headers and responses.

**The practice API in Postman.** With `python practice_api.py` running on your own computer, send
`GET http://127.0.0.1:8765/stations` from Postman, or import `http://127.0.0.1:8765/openapi.json`.

### An OpenAPI document, turned into requests

Postman's import reads an OpenAPI document and makes a request for every operation in it. Here is
the same idea in Python: read every documented operation, fill in its path parameters from the
documented examples, send it, check the status code against the ones the documentation lists, and
print the curl command a person would type.


In [13]:
for path, operations in doc["paths"].items():
    for method, operation in operations.items():
        url = doc["servers"][0]["url"] + path
        for parameter in operation.get("parameters", []):
            if parameter["in"] == "path":
                url = url.replace("{" + parameter["name"] + "}", parameter["example"])
        request = urllib.request.Request(url, method=method.upper())
        try:
            with urllib.request.urlopen(request) as response:
                status = response.status
        except HTTPError as error:
            status = error.code
        verdict = "as documented" if str(status) in operation["responses"] else "NOT documented"
        command = f"curl -i {url}" if method == "get" else f"curl -i -X {method.upper()} {url}"
        print(f"{method.upper()} {path:<16} {status}  {verdict:<15} {command}")


GET /stations        200  as documented   curl -i http://127.0.0.1:8765/stations
GET /stations/{id}   200  as documented   curl -i http://127.0.0.1:8765/stations/tromso


### Where each part came from

| In the loop | What it relies on | The section that showed it |
|---|---|---|
| `doc["paths"]`, then a method, then an operation | how an OpenAPI document is laid out | Reading an OpenAPI document |
| `doc["servers"][0]["url"]` | the base URL the document gives | Reading an OpenAPI document |
| `parameter["in"] == "path"` and `parameter["example"]` | a path parameter, with a documented example | Reading an OpenAPI document |
| `urllib.request.Request(url, method=...)` | a request with any method, as curl's `-X` sends | curl from a notebook cell |
| `except HTTPError` | an error status reaches `urlopen` as an exception | Reading real documentation: Open-Meteo |
| the printed curl command | a whole request, written as a single command | curl from a notebook cell |

Both documented operations answered as documented. On a larger API, the same loop is a quick check
that the documentation and the server still agree, and the **Schemas and Validation** notebook
extends it from status codes to the bodies themselves.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/02-exploring-an-api-solutions.ipynb).

**1.** Use curl to request `/stations/svalbard` with its status line and headers, and read the
`Content-Length`.


In [14]:
# your code here


**2.** Use curl with `-X` to send a `PATCH` request to `/stations/oslo`, and read which methods the
`Allow` header permits.


In [15]:
# your code here


**3.** Using `doc`, print every documented path with its methods, in the form `GET /stations`.


In [16]:
# your code here


**4.** Using `doc`, print the name, the location (`in`) and the `required` flag of every parameter
of `GET /stations/{id}`.


In [17]:
# your code here


**5.** The `200` response of `GET /stations` is an array. Use `resolve` on the schema of its items,
and print the fields a station summary must have.


In [18]:
# your code here


**6.** With curl, ask Open-Meteo for Bergen's three daily mean temperatures in Fahrenheit, for the
same dates, sending the request to `{OPEN_METEO}` and saving the body to a file with `-o`. Then
print the unit and the values with Python. Bergen is at latitude `60.39` and longitude `5.32`.


In [19]:
# your code here


## Common errors

### curl: (6) Could not resolve host: BASE, when the braces are left off


In [20]:
!curl -sS -i BASE/stations


curl: (6) Could not resolve host: BASE


Without braces, `BASE` is four letters of text: IPython replaced nothing, and curl went looking for
a computer called `BASE`. Note what did not happen: no exception, and the cell finished as though
it had worked. A shell command that fails in a notebook prints its error and carries on, so read
the output of every `!` line. `-sS` keeps curl silent except for errors, which is why the error is
all it printed.


In [21]:
!curl -sS -i {BASE}/stations


HTTP/1.1 200 OK
Server: PracticeAPI/1.0
Date: Sun, 01 Mar 2026 09:00:00 GMT
Content-Type: application/json
Content-Length: 144

[{"id": "bergen", "name": "Bergen"}, {"id": "oslo", "name": "Oslo"}, {"id": "svalbard", "name": "Svalbard"}, {"id": "tromso", "name": "Tromso"}]

### 000: curl's braces and IPython's braces in one line

`-w` takes a format in which `%{http_code}` stands for the status code. Next to `{BASE}`, it goes
wrong:


In [22]:
!curl -s -o /dev/null -w "%{http_code}\n" {BASE}/stations/atlantis


000


`000` is curl saying it never received a response. Before running a `!` line, IPython replaces every
`{...}` with the value of the Python expression inside. It could not evaluate `{http_code}`, which
belongs to curl, so it replaced nothing at all, and curl was given a URL whose host is the word
`BASE`. Double the braces that belong to curl, and IPython turns `{{` back into `{` on the way:


In [23]:
!curl -s -o /dev/null -w "%{{http_code}}\n" {BASE}/stations/atlantis


404


The Open-Meteo command in the worked examples doubles its braces for the same reason. In a
terminal, outside a notebook, write curl's braces single, as documentation shows them.

### KeyError: a real path, looked up in `paths`


In [24]:
doc["paths"]["/stations/tromso"]


KeyError: '/stations/tromso'

`paths` is keyed by the documented template, `/stations/{id}`, not by any URL you would send. Look
up the template, and put the id in when you build the request:


In [25]:
print(list(doc["paths"]))
print(doc["paths"]["/stations/{id}"]["get"]["summary"])


['/stations', '/stations/{id}']
Get one station


### KeyError: 'properties', when a $ref is read as a schema


In [26]:
schema = doc["paths"]["/stations/{id}"]["get"]["responses"]["200"]["content"]["application/json"]["schema"]
schema["properties"]


KeyError: 'properties'

The schema at that spot is `{'$ref': '#/components/schemas/Station'}`, a pointer with no
`properties` of its own. Follow it first:


In [27]:
print(list(resolve(doc, schema)["properties"]))


['id', 'name', 'latitude', 'longitude']


### The quiet one: a failed request that printed no error


In [28]:
!curl -s {BASE}/stations/atlantis


{"error": "no station with id 'atlantis'"}

A body, and nothing else: no status code, no error, and curl finished as though all was well. A
`404` is a complete response, and curl does not judge status codes unless asked, so without `-i` an
error body looks just like data. `-f`, for fail, makes curl treat a status of 400 or above as a
failure and print no body, and its exit status says what happened:


In [29]:
!curl -s -f {BASE}/stations/atlantis; echo "curl's exit status: $?"


curl's exit status: 22


`0` means success, and `22` is curl's exit status for an HTTP error under `-f`. Scripts check that
number to know whether a request worked.


### No error, and the wrong unit: a parameter Open-Meteo does not recognize


In [30]:
print(*daily_means(unit="fahrenheit"))


200 °C [4.6, 6.4, 7.1]


`200 OK`, and temperatures in Celsius. Open-Meteo has no parameter called `unit`, so it ignored it,
as most APIs do with a parameter they do not recognize, and nothing in the response says so except
the `°C` it reports. The documentation calls the parameter `temperature_unit`. Code that checks the
unit a response reports catches the mistake:


In [31]:
status, unit, values = daily_means(unit="fahrenheit")
if unit != "°F":
    print(f"asked for °F and got {unit}: check the parameter's name in the documentation")


asked for °F and got °C: check the parameter's name in the documentation


## Recap

- Explore an API before writing code against it: its documentation says what it accepts, and a
  request sent by hand shows what it returns.
- Most APIs ignore a parameter they do not recognize, so a misnamed parameter succeeds with the
  wrong answer. The documentation has the right name.
- `!` runs a line in the shell, and `{BASE}` puts a Python value into it. Double any braces that
  belong to the command, as in `%{{http_code}}`.
- curl: `-i` shows the status line and headers, `-s` keeps quiet, `-X` sets the method, `-G` with
  `--data-urlencode` builds a query, `-o` saves the body, `-w` prints details such as the status
  code, and `-f` fails on an error status.
- A failed shell command raises nothing in a notebook, so read its output.
- An OpenAPI document holds `paths`, then methods, then each operation's parameters and responses,
  with shared schemas under `components`, reached through `$ref`.
- Postman, Bruno and interactive documentation send the same requests as curl, and hand them back
  as curl or Python.
- Read the units and fields a response reports, rather than assuming them.


## What is next

The **Your First Request** notebook. Everything here was exploring. From there on, requests come
from Python, through the `requests` library, which sends a request in one call and returns a
response object that does most of the reading for you.


---

&#8592; **Previous:** [What an API Is](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/01-what-an-api-is.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Your First Request](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/03-your-first-request.ipynb) &#8594;
